In [ ]:
# ============================================================
# BRANCH A — DIRECT DOCUMENT REPRESENTATION
# D8 — Bhutan Land Management Project
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch A: Direct Ingestion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter

import hashlib
import json
import platform
import re
import subprocess
import sys

import pandas as pd

# antiword is used only for source-integrity diagnostics.
# The model receives the original legacy .doc file.
!apt-get update -qq
!apt-get install -y antiword -qq

In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D8"

DOCUMENT_NAME = (
    "World Bank — Bhutan - Land Management Project — "
    "Project Information Document (PID), Concept Stage"
)

BRANCH = "A"

BRANCH_NAME = "Direct Ingestion"

SOURCE_FORMAT = ".doc"

INPUT_REPRESENTATION = "Original legacy DOC"

DIRECT_DOCUMENT_INGESTION = True

EXPECTED_PHYSICAL_PAGE_COUNT = 4

EXPECTED_RECORD_COUNT = 49

EXPECTED_CATEGORY_COUNTS = {
    "Project metadata": 13,
    "Development issue": 10,
    "Bank rationale": 2,
    "Project objective": 3,
    "Project component": 3,
    "Safeguard policy": 6,
    "Financing": 7,
    "Contact information": 5
}


# ------------------------------------------------------------
# Fixed Stage 1 extraction schema
# ------------------------------------------------------------

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]


STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Qualifier",
    "Reporting Period",
    "Source Location"
]

VALUE_ALLOWED_TYPES = (
    str,
    int,
    float,
    type(None)
)

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Source Location"
]


ALLOWED_CATEGORIES = set(
    EXPECTED_CATEGORY_COUNTS
)

EXPECTED_QUALIFIER_VALUES = {
    "at least",
    "another",
    "less than",
    "some",
    "up to"
}


EXPECTED_SOURCE_MARKERS = {
    "textual_fraction_one_quarter":
        "one-quarter",

    "textual_fraction_one_third":
        "one-third",

    "source_typo_indigenous_pelple":
        "Indigenous Pelple",

    "source_typo_borrower_recipient":
        "BORROWER/RECEPIENT",

    "potential_environment_category":
        "F1",

    "report_number":
        "AB526",

    "project_id":
        "P087039"
}


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "outputs_D8_branch_A"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


INPUT_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D8_branch_A_input_integrity.json"
)

REPRESENTATION_PATH = (
    OUTPUT_DIR
    / "D8_branch_A_representation.json"
)

PROMPT_PATH = (
    OUTPUT_DIR
    / "D8_branch_A_prompt.txt"
)

RAW_RESPONSE_PATH = (
    OUTPUT_DIR
    / "D8_branch_A_raw_response.txt"
)

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D8_branch_A_parsed_extraction.json"
)

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D8_branch_A_technical_diagnostics.json"
)

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D8_branch_A_experiment_metadata.json"
)

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D8_branch_A_experiment_summary.json"
)


print(
    "Document:",
    DOCUMENT_ID
)

print(
    "Branch:",
    BRANCH
)

print(
    "Input representation:",
    INPUT_REPRESENTATION
)

print(
    "Expected reference records:",
    EXPECTED_RECORD_COUNT
)

print(
    "Expected fields:",
    len(EXPECTED_FIELDS)
)

In [ ]:
# ============================================================
# 2. Source document and integrity diagnostics
# ============================================================

print(
    "Upload the original D8 legacy Word document (.doc)."
)


uploaded = files.upload()


doc_paths = [
    Path(name)

    for name
    in uploaded

    if name.lower().endswith(
        ".doc"
    )
]


if len(doc_paths) != 1:

    raise ValueError(
        "Upload exactly one legacy .doc source document."
    )


SOURCE_PATH = doc_paths[0]


# ------------------------------------------------------------
# File hashing
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)


FILE_SIZE_BYTES = (
    SOURCE_PATH.stat().st_size
)

FILE_NON_EMPTY = (
    FILE_SIZE_BYTES > 0
)


# ------------------------------------------------------------
# Diagnostic legacy-DOC text extraction
# ------------------------------------------------------------

def extract_text_from_legacy_doc(path):

    result = subprocess.run(
        [
            "antiword",
            str(path)
        ],
        capture_output=True,
        text=True,
        errors="replace"
    )


    if result.returncode != 0:

        raise RuntimeError(
            "antiword could not extract the D8 source document.\n"
            f"stderr: {result.stderr}"
        )


    if not result.stdout.strip():

        raise RuntimeError(
            "The D8 source document produced no extractable text."
        )


    return result.stdout


SOURCE_TEXT = extract_text_from_legacy_doc(
    SOURCE_PATH
)


TEXT_EXTRACTABLE = bool(
    SOURCE_TEXT.strip()
)


OCR_REQUIRED = False


PAGE_BOUNDARIES_AVAILABLE_IN_ANTIWORD = (
    "\f"
    in SOURCE_TEXT
)


# ------------------------------------------------------------
# Expected source-component diagnostics
# ------------------------------------------------------------

EXPECTED_SECTION_PATTERNS = {
    "pid_header":
        r"PROJECT\s+INFORMATION\s+DOCUMENT\s*\(PID\)",

    "concept_stage":
        r"CONCEPT\s+STAGE",

    "development_issues":
        r"1\.\s+Key\s+development\s+issues",

    "proposed_objectives":
        r"2\.\s+Proposed\s+objective\(s\)",

    "preliminary_description":
        r"3\.\s+Preliminary\s+description",

    "safeguards":
        r"4\.\s+Safeguard\s+Policies",

    "financing":
        r"5\.\s+Tentative\s+financing",

    "contact":
        r"6\.\s+Contact\s+point"
}


SECTION_MARKER_STATUS = {
    marker: bool(
        re.search(
            pattern,
            SOURCE_TEXT,
            flags=(
                re.IGNORECASE
                | re.MULTILINE
            )
        )
    )

    for marker, pattern
    in EXPECTED_SECTION_PATTERNS.items()
}


SECTION_MARKERS_VALID = all(
    SECTION_MARKER_STATUS.values()
)


CHARACTER_COUNT = len(
    SOURCE_TEXT
)


WORD_COUNT = len(
    SOURCE_TEXT.split()
)


NUMERIC_TOKEN_COUNT = len(
    re.findall(
        r"(?<!\w)"
        r"[£$]?"
        r"\(?-?\d[\d,]*"
        r"(?:\.\d+)?%?"
        r"\)?",
        SOURCE_TEXT
    )
)


# ------------------------------------------------------------
# Direct-ingestion suitability
# ------------------------------------------------------------

DIRECT_DOC_INGESTION_USABLE = all([
    FILE_NON_EMPTY,
    TEXT_EXTRACTABLE,
    SECTION_MARKERS_VALID
])


INPUT_INTEGRITY_PASSED = (
    DIRECT_DOC_INGESTION_USABLE
)


INPUT_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_file":
        SOURCE_PATH.name,

    "input_file_sha256":
        SOURCE_SHA256,

    "input_representation":
        INPUT_REPRESENTATION,

    "source_format":
        SOURCE_FORMAT,

    "legacy_binary_word_format":
        True,

    "file_size_bytes":
        FILE_SIZE_BYTES,

    "file_non_empty":
        FILE_NON_EMPTY,

    "expected_physical_page_count":
        EXPECTED_PHYSICAL_PAGE_COUNT,

    "physical_page_count_programmatically_verified":
        False,

    "physical_page_locations_manually_verified_in_stage1":
        True,

    "page_boundaries_available_in_antiword_text":
        PAGE_BOUNDARIES_AVAILABLE_IN_ANTIWORD,

    "text_extractable":
        TEXT_EXTRACTABLE,

    "ocr_required":
        OCR_REQUIRED,

    "character_count":
        CHARACTER_COUNT,

    "word_count":
        WORD_COUNT,

    "numeric_token_count":
        NUMERIC_TOKEN_COUNT,

    "section_marker_checks":
        SECTION_MARKER_STATUS,

    "all_expected_components_present":
        SECTION_MARKERS_VALID,

    "direct_doc_ingestion_usable":
        DIRECT_DOC_INGESTION_USABLE,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED
}


INPUT_INTEGRITY_PATH.write_text(
    json.dumps(
        INPUT_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    "Source:",
    SOURCE_PATH.name
)

print(
    "Source size:",
    f"{FILE_SIZE_BYTES:,} bytes"
)

print(
    "SHA-256:",
    SOURCE_SHA256
)

print(
    "Text extractable:",
    TEXT_EXTRACTABLE
)

print(
    "antiword page boundaries available:",
    PAGE_BOUNDARIES_AVAILABLE_IN_ANTIWORD
)

print(
    "\nSection markers:"
)

print(
    json.dumps(
        SECTION_MARKER_STATUS,
        ensure_ascii=False,
        indent=2
    )
)

print(
    "\nInput integrity passed:",
    INPUT_INTEGRITY_PASSED
)


# Source/integrity failures may stop the notebook.

if not FILE_NON_EMPTY:

    raise AssertionError(
        "The D8 source file is empty."
    )


if not TEXT_EXTRACTABLE:

    raise AssertionError(
        "The D8 source document does not "
        "contain extractable text."
    )


if not SECTION_MARKERS_VALID:

    raise AssertionError(
        "One or more required D8 source "
        "sections were not detected."
    )

In [ ]:
# ============================================================
# 3. Branch A representation
# ============================================================

REPRESENTATION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        "Original source document",

    "input_file":
        SOURCE_PATH.name,

    "input_format":
        SOURCE_FORMAT,

    "legacy_binary_word_format":
        True,

    "diagnostic_text_extraction_applied":
        True,

    "diagnostic_text_extraction_tool":
        "antiword",

    "diagnostic_text_used_as_model_input":
        False,

    "doc_to_text_conversion_applied_for_model_input":
        False,

    "derived_representation_used_as_model_input":
        False,

    "ocr_applied":
        False,

    "document_reconstruction_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "source_spelling_correction_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "complete_original_document_supplied":
        True,

    "model_input_description": (
        "The complete original D8 legacy binary DOC is "
        "submitted directly to the LLM. antiword text "
        "extraction is used only for source-integrity "
        "diagnostics and is not supplied to the model "
        "as an alternative representation."
    )
}


REPRESENTATION_PATH.write_text(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 4. Extraction prompt
# ============================================================

BRANCH_A_PROMPT = """You are an information extraction assistant.

Extract the project-information records represented within the defined
scope of the attached original legacy Word document:

“Bhutan - Land Management Project”
Project Information Document (PID), Concept Stage.

Treat the attached original document as the only source of information.

Include records from the following defined source regions.

1. Project metadata

Extract one record for each of these labelled header fields:

- Report No.
- Project Name
- Region
- Sector
- Project ID
- GEF Focal Area
- Borrower(s)
- Implementing Agency
- Environment Category
- Safeguard Classification
- Date PID Prepared
- Estimated Date of Appraisal Authorization
- Estimated Date of Board Approval

For the Sector field, preserve the complete represented sector text as
a single Value. Do not split its embedded percentages into additional
records.

For checkbox fields, extract the selected category represented by the
document.

Do not create separate records from implementing-agency telephone
numbers embedded within the Implementing Agency field.

2. Development issues

Within Section 1, “Key development issues and rationale for Bank
involvement”, extract the predefined quantitative development
observations concerning:

- the long-term forest-cover policy requirement;
- the share of country area set aside for protected areas;
- the additional area offered for wildlife corridors;
- population density per square kilometre of arable land;
- the urban growth rate;
- arable land as a share of land area;
- agricultural land affected by water erosion;
- the contribution of hydropower revenue to the development budget;
- villages not connected to feeder roads;
- villages facing food insecurity.

Do not extract comparison values concerning other world regions as
separate observations.

3. Bank rationale

Within the “Rationale for Bank Involvement” subsection, extract the
principal qualitative records concerning:

- limitations of the existing sector-oriented institutional framework
  in providing cross-sectoral accountability and incentive mechanisms;
- the governance, political-will and environmental-stewardship factors
  supporting Bhutan's suitability for GEF support.

Do not create additional records from illustrative examples or
supporting narrative details.

4. Project objectives

Within Section 2, “Proposed objective(s)”, extract each principal
project-objective statement represented by the source.

The scope consists of the objective to promote sustainable-land-
management mechanisms, the objective concerning technical innovations,
ecosystem functions and cross-sectoral mechanisms, and the objective
concerning multi-sectoral land and watershed planning with local
participation.

5. Project components

Within Section 3, “Preliminary description”, extract one record for
each explicitly labelled project component.

For each component:

- preserve the component number and component name;
- preserve a concise source-grounded description of the component;
- extract the explicitly represented estimated cost as Value;
- preserve the represented monetary scale in Unit;
- preserve any explicit approximation, ceiling or threshold wording
  associated with the cost in Qualifier.

Do not calculate a total component cost.

6. Safeguard policies

Within Section 4, “Safeguard Policies that Might Apply”, extract:

- each explicitly listed safeguard-policy item;
- the currently assessed environmental-assessment category;
- the potential environmental-assessment category described for
  community sub-project grants with environmental implications.

Preserve source codes, labels and represented spellings exactly as
shown. Do not silently repair typographical errors or category codes.

7. Tentative financing

Within Section 5, “Tentative financing”, extract:

- one record for every explicitly represented financing-source row;
- the explicitly represented Total row.

Preserve the source labels exactly as represented.

Use the represented monetary scale as Unit.

Do not calculate or recompute the Total.

8. Contact information

Within Section 6, “Contact point”, extract one record for each labelled
contact field:

- Contact
- Title
- Tel
- Fax
- Email

Preserve phone numbers and email addresses as JSON strings.

For every included record extract exactly these fields:

- Category
- Topic
- Description
- Value
- Unit
- Qualifier
- Reporting Period
- Source Location

Category:

Use exactly one of:

- Project metadata
- Development issue
- Bank rationale
- Project objective
- Project component
- Safeguard policy
- Financing
- Contact information

Topic:

- Preserve the relevant source-grounded project field, issue,
  objective, component, policy, financing source or contact item.
- Do not merge distinct source observations.

Description:

- Provide a concise source-grounded description of the represented
  record.
- Do not add external interpretation.

Value:

- Use a JSON number for explicitly represented numeric values.
- Use a JSON string for explicitly represented textual values, codes,
  dates, telephone numbers, email addresses, textual fractions or
  category labels.
- Use null when no separate Value is represented.
- Preserve textual fractional quantities in their represented textual
  form rather than converting them to numeric percentages.
- Do not derive separate numbers from percentages embedded within a
  complete textual field.
- Do not calculate, infer, derive, rescale or convert values.

Unit:

- Preserve the explicitly associated measurement unit or scale.
- Use null when no explicit unit applies.
- Do not place approximation, threshold or inequality wording in Unit.

Qualifier:

- Preserve explicit source qualifiers or modifiers associated with a
  Value in this field.
- This includes approximation, threshold, extent or ceiling wording
  represented by the source.
- Use null when no explicit qualifier applies.
- Do not merge Qualifier wording into Unit.

Reporting Period:

- Preserve explicitly associated dates or periods.
- Use null when no separate reporting period is explicitly associated
  with the record.

Source Location:

Use concise physical-DOC locations grounded in the original document,
for example:

- DOC page 1 — Header
- DOC page 1 — Section 1, Key Development Issues
- DOC page 2 — Section 1, Key Development Issues
- DOC page 2 — Section 1, Rationale for Bank Involvement
- DOC page 3 — Section 2, Proposed objective(s)
- DOC page 3 — Section 3, Preliminary description
- DOC page 4 — Section 4, Safeguard Policies that Might Apply
- DOC page 4 — Section 5, Tentative financing
- DOC page 4 — Section 6, Contact point

Additional extraction rules:

- Use only information explicitly represented in the original source.
- Preserve source wording, codes, labels and spellings where relevant.
- Preserve represented typographical errors rather than silently
  correcting them.
- Preserve repeated observations if the fixed scope explicitly
  requires them in distinct source locations.
- Do not use external knowledge.
- Do not follow external links.
- Do not calculate or infer missing information.
- Do not repair source values or codes.
- Do not convert units.
- Do not add explanatory examples from narrative text outside the
  defined extraction scope.
- Verify that all content within the defined source scope has been
  processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{
  "document_id": "D8",
  "branch": "A",
  "records": [
    {
      "Category": null,
      "Topic": null,
      "Description": null,
      "Value": null,
      "Unit": null,
      "Qualifier": null,
      "Reporting Period": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
"""


PROMPT_PATH.write_text(
    BRANCH_A_PROMPT,
    encoding="utf-8"
)


PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)


print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)

print()

print(
    BRANCH_A_PROMPT
)

## Independent Branch A extraction

Open a new independent conversation.

Upload:

1. the complete original D8 legacy file;
2. `D8_branch_A_prompt.txt`.

Submit the prompt once.

Save the complete, untouched model response as:

`D8_branch_A_raw_response.txt`


In [ ]:
# ============================================================
# 5. Raw response preservation and parsing
# ============================================================

print(
    "Upload the untouched "
    "D8_branch_A_raw_response.txt file."
)


uploaded = files.upload()


txt_paths = [
    Path(name)

    for name
    in uploaded

    if name.lower().endswith(
        ".txt"
    )
]


if len(txt_paths) != 1:

    raise ValueError(
        "Upload exactly one TXT raw-response file."
    )


UPLOADED_RAW_RESPONSE_PATH = (
    txt_paths[0]
)


raw_response_text = (
    UPLOADED_RAW_RESPONSE_PATH.read_text(
        encoding="utf-8"
    )
)


if not raw_response_text.strip():

    raise ValueError(
        "The uploaded raw response is empty."
    )


# ------------------------------------------------------------
# Preserve untouched raw response BEFORE parsing
# ------------------------------------------------------------

RAW_RESPONSE_PATH.write_text(
    raw_response_text,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)


# ------------------------------------------------------------
# Parse without crashing on invalid JSON
# ------------------------------------------------------------

valid_json = False

json_parsing_error = None

parsed_response = None


try:

    parsed_response = json.loads(
        raw_response_text
    )

    valid_json = True


except json.JSONDecodeError as error:

    json_parsing_error = str(
        error
    )


# ------------------------------------------------------------
# Validate standardized top-level wrapper
# ------------------------------------------------------------

top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)


document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_response
)


document_id_correct = (
    top_level_object_valid
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)


branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_response
)


branch_correct = (
    top_level_object_valid
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)


records_present = (
    top_level_object_valid
    and "records"
    in parsed_response
)


records_is_list = (
    top_level_object_valid
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)


# ------------------------------------------------------------
# Record-level evaluation is possible whenever a valid
# ------------------------------------------------------------

records_evaluable = (
    valid_json
    and top_level_object_valid
    and records_present
    and records_is_list
)


if records_evaluable:

    extracted_records = (
        parsed_response[
            "records"
        ]
    )

    observed_record_count = len(
        extracted_records
    )


else:

    extracted_records = []

    observed_record_count = None


# ------------------------------------------------------------
# Create parsed extraction only when records are evaluable
# ------------------------------------------------------------

parsed_extraction_created = False

parsed_extraction_sha256 = None


if records_evaluable:

    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get(
                "document_id"
            ),

        "branch":
            parsed_response.get(
                "branch"
            ),

        "records":
            extracted_records
    }


    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    parsed_extraction_created = True

    parsed_extraction_sha256 = (
        sha256_file(
            PARSED_EXTRACTION_PATH
        )
    )


print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)

print(
    "Valid JSON:",
    valid_json
)

print(
    "JSON parsing error:",
    json_parsing_error
)

print(
    "Top-level object valid:",
    top_level_object_valid
)

print(
    "Document ID correct:",
    document_id_correct
)

print(
    "Branch correct:",
    branch_correct
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed record count:",
    observed_record_count
)


if records_evaluable:

    extracted_df = pd.DataFrame(
        extracted_records
    )

    display(
        extracted_df.head(
            12
        )
    )

In [ ]:
# ============================================================
# 6. Record and content diagnostics
# ============================================================

record_structure_issues = []

field_type_issues = []

missing_mandatory_values = []


# ------------------------------------------------------------
# A. Record schema
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        "Record is not a JSON object"
                }
            )

            continue


        observed_fields = list(
            record.keys()
        )


        if (
            observed_fields
            != EXPECTED_FIELDS
        ):

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        (
                            "Field names or field "
                            "order differ"
                        ),

                    "expected_fields":
                        EXPECTED_FIELDS,

                    "observed_fields":
                        observed_fields,

                    "missing_fields":
                        [
                            field

                            for field
                            in EXPECTED_FIELDS

                            if field
                            not in record
                        ],

                    "extra_fields":
                        [
                            field

                            for field
                            in observed_fields

                            if field
                            not in EXPECTED_FIELDS
                        ]
                }
            )


    records_with_structure_issues = len({
        issue[
            "record_index"
        ]

        for issue
        in record_structure_issues
    })


    record_schema_valid = (
        records_with_structure_issues
        == 0
    )


else:

    records_with_structure_issues = None

    record_schema_valid = None


# ------------------------------------------------------------
# B. Field types
#
# Stage 1:
#
# textual fields -> string or null
# Value          -> string, number or null
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):

            continue


        for field in STRING_OR_NULL_FIELDS:

            value = record.get(
                field
            )


            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):

                field_type_issues.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field,

                        "observed_type":
                            type(
                                value
                            ).__name__,

                        "expected_type":
                            "string or null"
                    }
                )


        value = record.get(
            "Value"
        )


        if (
            isinstance(
                value,
                bool
            )
            or not isinstance(
                value,
                VALUE_ALLOWED_TYPES
            )
        ):

            field_type_issues.append(
                {
                    "record_index":
                        record_index,

                    "field":
                        "Value",

                    "observed_type":
                        type(
                            value
                        ).__name__,

                    "expected_type":
                        "string, number or null"
                }
            )


        # ----------------------------------------------------
        # Mandatory-content diagnostic
        # ----------------------------------------------------

        for field in MANDATORY_CONTENT_FIELDS:

            value = record.get(
                field
            )


            if (
                value is None
                or value == ""
            ):

                missing_mandatory_values.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field
                    }
                )


    records_with_type_issues = len({
        issue[
            "record_index"
        ]

        for issue
        in field_type_issues
    })


    field_types_valid = (
        records_with_type_issues
        == 0
    )


    missing_mandatory_value_count = len(
        missing_mandatory_values
    )


    mandatory_fields_complete = (
        missing_mandatory_value_count
        == 0
    )


else:

    records_with_type_issues = None

    field_types_valid = None

    missing_mandatory_value_count = None

    mandatory_fields_complete = None


# ------------------------------------------------------------
# C. Record-count and category diagnostics
# ------------------------------------------------------------

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    observed_category_counts = dict(
        Counter(
            record.get(
                "Category"
            )

            for record
            in extracted_records

            if isinstance(
                record,
                dict
            )
        )
    )


    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


else:

    record_count_valid = None

    observed_category_counts = None

    categories_valid = None

    category_counts_valid = None


# ------------------------------------------------------------
# D. Exact complete-record duplicate diagnostic
# ------------------------------------------------------------

if records_evaluable:

    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(
                    field
                ),
                ensure_ascii=False,
                sort_keys=True
            )

            for field
            in EXPECTED_FIELDS
        )

        for record
        in extracted_records

        if isinstance(
            record,
            dict
        )
    )


    duplicate_records = [
        list(
            key
        )

        for key, count
        in duplicate_counter.items()

        if count > 1
    ]


    duplicate_record_count = len(
        duplicate_records
    )


    duplicate_records_absent = (
        duplicate_record_count
        == 0
    )


else:

    duplicate_records = None

    duplicate_record_count = None

    duplicate_records_absent = None


# ------------------------------------------------------------
# E. Value-type diagnostics
# ------------------------------------------------------------

if records_evaluable:

    numeric_value_count = sum(
        1

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and isinstance(
                record.get(
                    "Value"
                ),
                (
                    int,
                    float
                )
            )

            and not isinstance(
                record.get(
                    "Value"
                ),
                bool
            )
        )
    )


    text_value_count = sum(
        1

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and isinstance(
                record.get(
                    "Value"
                ),
                str
            )
        )
    )


    null_value_count = sum(
        1

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and record.get(
                "Value"
            )
            is None
        )
    )


else:

    numeric_value_count = None

    text_value_count = None

    null_value_count = None


# ------------------------------------------------------------
# F. Qualifier diagnostics
# ------------------------------------------------------------

if records_evaluable:

    observed_qualifier_values = sorted({
        record.get(
            "Qualifier"
        ).casefold()

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and isinstance(
                record.get(
                    "Qualifier"
                ),
                str
            )

            and record.get(
                "Qualifier"
            ).strip()
        )
    })


    qualifier_count = sum(
        1

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and record.get(
                "Qualifier"
            )
            is not None
        )
    )


    expected_qualifier_presence = {
        qualifier: (
            qualifier.casefold()
            in observed_qualifier_values
        )

        for qualifier
        in EXPECTED_QUALIFIER_VALUES
    }


    expected_qualifiers_preserved = all(
        expected_qualifier_presence.values()
    )


    qualifier_embedded_in_unit_records = [
        {
            "record_index":
                record_index,

            "Unit":
                record.get(
                    "Unit"
                )
        }

        for record_index, record
        in enumerate(
            extracted_records
        )

        if (
            isinstance(
                record,
                dict
            )

            and isinstance(
                record.get(
                    "Unit"
                ),
                str
            )

            and any(
                qualifier.casefold()
                in record.get(
                    "Unit"
                ).casefold()

                for qualifier
                in EXPECTED_QUALIFIER_VALUES
            )
        )
    ]


    qualifier_embedded_in_unit_count = len(
        qualifier_embedded_in_unit_records
    )


    qualifier_not_embedded_in_unit = (
        qualifier_embedded_in_unit_count
        == 0
    )


    component_up_to_records = [
        record

        for record
        in extracted_records

        if (
            isinstance(
                record,
                dict
            )

            and record.get(
                "Category"
            )
            == "Project component"

            and isinstance(
                record.get(
                    "Qualifier"
                ),
                str
            )

            and record.get(
                "Qualifier"
            ).casefold()
            == "up to"
        )
    ]


    component_up_to_count = len(
        component_up_to_records
    )


    all_component_qualifiers_preserved = (
        component_up_to_count
        == 3
    )


else:

    observed_qualifier_values = None

    qualifier_count = None

    expected_qualifier_presence = None

    expected_qualifiers_preserved = None

    qualifier_embedded_in_unit_records = None

    qualifier_embedded_in_unit_count = None

    qualifier_not_embedded_in_unit = None

    component_up_to_count = None

    all_component_qualifiers_preserved = None


# ------------------------------------------------------------
# G. Source-preservation diagnostics
# ------------------------------------------------------------

if records_evaluable:

    extraction_search_text = json.dumps(
        extracted_records,
        ensure_ascii=False
    )


    source_marker_status = {
        marker: (
            expected_text
            in extraction_search_text
        )

        for marker, expected_text
        in EXPECTED_SOURCE_MARKERS.items()
    }


    source_markers_preserved = all(
        source_marker_status.values()
    )


    textual_quantities_preserved = all([
        source_marker_status[
            "textual_fraction_one_quarter"
        ],

        source_marker_status[
            "textual_fraction_one_third"
        ]
    ])


    source_typos_preserved = all([
        source_marker_status[
            "source_typo_indigenous_pelple"
        ],

        source_marker_status[
            "source_typo_borrower_recipient"
        ],

        source_marker_status[
            "potential_environment_category"
        ]
    ])


else:

    source_marker_status = None

    source_markers_preserved = None

    textual_quantities_preserved = None

    source_typos_preserved = None


# ------------------------------------------------------------
# H. Explicit financing-total diagnostic
#
# Reference-side diagnostic only.
# ------------------------------------------------------------

if records_evaluable:

    financing_total_present = any(
        (
            isinstance(
                record,
                dict
            )

            and record.get(
                "Category"
            )
            == "Financing"

            and isinstance(
                record.get(
                    "Topic"
                ),
                str
            )

            and record.get(
                "Topic"
            ).casefold()
            == "total"

            and record.get(
                "Value"
            )
            == 16.5
        )

        for record
        in extracted_records
    )


else:

    financing_total_present = None


# ------------------------------------------------------------
# Content diagnostics
# ------------------------------------------------------------

CONTENT_DIAGNOSTICS = {
    "record_count_matches_reference":
        record_count_valid,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records_absent":
        duplicate_records_absent,

    "numeric_value_count":
        numeric_value_count,

    "text_value_count":
        text_value_count,

    "null_value_count":
        null_value_count,

    "qualifier_count":
        qualifier_count,

    "observed_qualifier_values":
        observed_qualifier_values,

    "expected_qualifier_presence":
        expected_qualifier_presence,

    "expected_qualifiers_preserved":
        expected_qualifiers_preserved,

    "qualifier_embedded_in_unit_count":
        qualifier_embedded_in_unit_count,

    "qualifier_not_embedded_in_unit":
        qualifier_not_embedded_in_unit,

    "component_up_to_count":
        component_up_to_count,

    "all_component_qualifiers_preserved":
        all_component_qualifiers_preserved,

    "source_marker_status":
        source_marker_status,

    "source_markers_preserved":
        source_markers_preserved,

    "textual_quantities_preserved":
        textual_quantities_preserved,

    "source_typos_preserved":
        source_typos_preserved,

    "financing_total_present":
        financing_total_present
}


print(
    "Record schema valid:",
    record_schema_valid
)

print(
    "Records with structure issues:",
    records_with_structure_issues
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Records with type issues:",
    records_with_type_issues
)

print(
    "Observed records:",
    observed_record_count
)

print(
    "Record count matches reference:",
    record_count_valid
)

print(
    "Category counts match reference:",
    category_counts_valid
)

print(
    "Missing mandatory values:",
    missing_mandatory_value_count
)

print(
    "Duplicate complete records:",
    duplicate_record_count
)

print(
    "Qualifiers observed:",
    qualifier_count
)

print(
    "Qualifier wording embedded in Unit:",
    qualifier_embedded_in_unit_count
)

print(
    "All project-component 'up to' qualifiers preserved:",
    all_component_qualifiers_preserved
)

print(
    "Source markers preserved:",
    source_markers_preserved
)

print(
    "Financing total preserved:",
    financing_total_present
)

print(
    "\nObserved category counts:"
)

print(
    json.dumps(
        observed_category_counts,
        ensure_ascii=False,
        indent=2
    )
    if observed_category_counts is not None
    else None
)

print(
    "\nObserved qualifier values:"
)

print(
    json.dumps(
        observed_qualifier_values,
        ensure_ascii=False,
        indent=2
    )
    if observed_qualifier_values is not None
    else None
)

In [ ]:
# ============================================================
# 7. Technical diagnostic summary and experiment metadata
# ============================================================

# ------------------------------------------------------------
# Technical/schema validity only
# ------------------------------------------------------------

STRUCTURAL_CHECKS = {
    "valid_json":
        bool(
            valid_json
        ),

    "top_level_object_valid":
        bool(
            top_level_object_valid
        ),

    "document_id_present":
        bool(
            document_id_present
        ),

    "document_id_correct":
        bool(
            document_id_correct
        ),

    "branch_present":
        bool(
            branch_present
        ),

    "branch_correct":
        bool(
            branch_correct
        ),

    "records_present":
        bool(
            records_present
        ),

    "records_is_list":
        bool(
            records_is_list
        ),

    "record_schema_valid":
        (
            record_schema_valid
            if records_evaluable
            else None
        ),

    "field_types_valid":
        (
            field_types_valid
            if records_evaluable
            else None
        )
}


structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])


# ------------------------------------------------------------
# Structure check artifact
# ------------------------------------------------------------

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,

    "valid_json":
        valid_json,

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        top_level_object_valid,

    "document_id_present":
        document_id_present,

    "document_id_correct":
        document_id_correct,

    "branch_present":
        branch_present,

    "branch_correct":
        branch_correct,

    "records_present":
        records_present,

    "records_is_list":
        records_is_list,

    "records_evaluable":
        records_evaluable,

    "structural_checks":
        STRUCTURAL_CHECKS,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_valid":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_valid":
        category_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issue_count":
        (
            len(
                field_type_issues
            )
            if records_evaluable
            else None
        ),

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "missing_mandatory_values":
        (
            missing_mandatory_values
            if records_evaluable
            else None
        ),

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records":
        duplicate_records,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        )
}


TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Experiment metadata
# ------------------------------------------------------------

EXPERIMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "source_structure": {
        "expected_physical_page_count":
            EXPECTED_PHYSICAL_PAGE_COUNT,

        "physical_page_count_programmatically_verified":
            False,

        "physical_page_locations_manually_verified_in_stage1":
            True,

        "page_boundaries_available_in_antiword_text":
            PAGE_BOUNDARIES_AVAILABLE_IN_ANTIWORD,

        "text_extractable":
            TEXT_EXTRACTABLE,

        "expected_components_verified":
            SECTION_MARKERS_VALID
    },

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "legacy_binary_word_format":
        True,

    "diagnostic_text_extraction_applied":
        True,

    "diagnostic_text_extraction_tool":
        "antiword",

    "diagnostic_text_used_as_model_input":
        False,

    "doc_to_text_conversion_applied_for_model_input":
        False,

    "derived_representation_used_as_model_input":
        False,

    "ocr_applied":
        False,

    "structural_conversion_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_conversion_applied":
        False,

    "source_spelling_correction_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "expected_extraction_scope": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,

        "expected_category_counts":
            EXPECTED_CATEGORY_COUNTS,

        "expected_fields":
            EXPECTED_FIELDS
    },

    "reference_expectations_disclosed_to_model":
        False,

    "input_integrity_file":
        INPUT_INTEGRITY_PATH.name,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED,

    "representation_file":
        REPRESENTATION_PATH.name,

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "expected_output_format":
        (
            "JSON object with document_id, "
            "branch and records"
        ),

    "execution_environment":
        "Independent ChatGPT conversation",

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        ),

    "notes": (
        "Branch A submits the complete original D8 legacy "
        "binary DOC directly to the model. antiword text "
        "extraction is used only for source-integrity "
        "diagnostics and is not supplied to the model. "
        "No DOC-to-text model input conversion, OCR, "
        "structural conversion, normalisation, semantic "
        "rewriting, unit conversion, source correction or "
        "manual reconstruction is applied before extraction. "
        "Stage 1 reference values, expected record count and "
        "expected category distribution are not supplied to "
        "the model. Content-level validation is performed separately in "
        "Validation A — D8."
    )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Experiment summary
# ------------------------------------------------------------

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        INPUT_INTEGRITY_PASSED,

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_count":
        duplicate_record_count,

    "numeric_value_count":
        numeric_value_count,

    "text_value_count":
        text_value_count,

    "null_value_count":
        null_value_count,

    "qualifier_count":
        qualifier_count,

    "qualifier_embedded_in_unit_count":
        qualifier_embedded_in_unit_count,

    "all_component_qualifiers_preserved":
        all_component_qualifiers_preserved,

    "textual_quantities_preserved":
        textual_quantities_preserved,

    "source_typos_preserved":
        source_typos_preserved,

    "source_markers_preserved":
        source_markers_preserved,

    "financing_total_present":
        financing_total_present,

    "raw_response_preserved":
        RAW_RESPONSE_PATH.exists(),

    "parsed_extraction_created":
        parsed_extraction_created,

    "content_validation_performed":
        False,

    "notes": (
        "This notebook performs source verification, "
        "D8 Branch A direct-DOC execution preservation, "
        "technical/schema checks and document-specific "
        "content diagnostics only. Agreement with the fixed "
        "Stage 1 reference dataset is evaluated separately "
        "in Validation A — D8."
    )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print(
    "Structural checks:"
)

print(
    json.dumps(
        STRUCTURAL_CHECKS,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\nContent diagnostics:"
)

print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\nStructurally evaluable:",
    structurally_evaluable
)


print(
    "\nExperiment summary:"
)

print(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\n" + "=" * 60
)

print(
    "D8 Branch A experiment completed"
)

print(
    "=" * 60
)


print(
    "Input integrity passed       :",
    INPUT_INTEGRITY_PASSED
)

print(
    "Raw response preserved       :",
    RAW_RESPONSE_PATH.exists()
)

print(
    "Valid JSON                   :",
    valid_json
)

print(
    "Records evaluable            :",
    records_evaluable
)

print(
    "Expected records             :",
    EXPECTED_RECORD_COUNT
)

print(
    "Observed records             :",
    (
        observed_record_count
        if observed_record_count
        is not None
        else "Not evaluable"
    )
)

print(
    "Record count matches         :",
    record_count_valid
)

print(
    "Category counts match        :",
    category_counts_valid
)

print(
    "Record schema valid          :",
    record_schema_valid
)

print(
    "Field types valid            :",
    field_types_valid
)

print(
    "Structurally evaluable              :",
    structurally_evaluable
)

print(
    "Content validation performed : False"
)

print(
    "Next step                    : Validation A — D8"
)


# ------------------------------------------------------------
# Output existence checks
#
# Invalid model JSON is allowed as an experimental result, so
# parsed_extraction.json is required only when records_evaluable.
# ------------------------------------------------------------

required_output_paths = [
    INPUT_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    PROMPT_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]


if (
    parsed_extraction_created
    and PARSED_EXTRACTION_PATH.exists()
):

    required_output_paths.append(
        PARSED_EXTRACTION_PATH
    )


missing_output_files = [
    path.name

    for path
    in required_output_paths

    if not path.exists()
]


if missing_output_files:

    raise AssertionError(
        "Missing output files: "
        f"{missing_output_files}"
    )


print(
    "\nGenerated D8 Branch A files:\n"
)


for path in required_output_paths:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )